# Importações

In [2]:
import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import re

# RF01 – Criar ou Carregar o Dataset de Vendas

In [ ]:
def gerar_dataset_vendas(n_registros=150, seed=42):
    """
    Gera um dataset sintético de vendas com dados propositalmente sujos,
    incluindo valores nulos, strings sujas, datas inválidas e outliers.
    """

    random.seed(seed)
    np.random.seed(seed)

    produtos = [
        'Notebook',
        'Smartphone',
        'Tablet',
        'Monitor',
        'Teclado',
        'Mouse'
    ]

    precos = {
        'Notebook': 3500,
        'Smartphone': 2200,
        'Tablet': 1800,
        'Monitor': 1200,
        'Teclado': 250,
        'Mouse': 120
    }

    categorias = {
        "Notebook": "Computadores",
        "Smartphone": "Celulares",
        "Tablet": "Celulares",
        "Monitor": "Computadores",
        "Teclado": "Periféricos",
        "Mouse": "Periféricos"
    }

    regioes = [
        "Sudeste",
        "Sul",
        "Nordeste",
        "Centro-Oeste",
        "Norte"
    ]

    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]

    data_inicio = datetime(2024, 1, 1)

    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]

        data = data_inicio + timedelta(days=random.randint(0, 364))

        # Inserindo dados intencionalmente sujos para limpeza
        if random.random() < 0.05:
            quantidade = None  # valor nulo

        if random.random() < 0.04:
            preco = None  # valor nulo

        if random.random() < 0.03:
            produto = " " + produto  # espaço extra (string suja)

        data_str = (
            data.strftime("%Y-%m-%d")
            if random.random() > 0.02
            else "DATA INVALIDA"
        )

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })

    return pd.DataFrame(dados)


# Gerar e salvar
df_bruto = gerar_dataset_vendas()
os.makedirs("../data/raw", exist_ok=True)
df_bruto.to_csv("../data/raw/vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")


In [ ]:
df_bruto

# RF02 – Inspecionar e Descrever os Dados

In [7]:
df = df_bruto

In [ ]:
def inspecionar_dados(df):
  """Exibe informações básicas do df."""
  print("\n=== INSPEÇÃO INICIAL DO DATASET ===" )
  print(f"Shape: {df.shape}")
  print(f"\nColunas: {list(df.columns)}" )
  print(f"\nTipos de dados:\n{df.dtypes}" )
  print(f"\nValores nulos por coluna:\n{df.isnull().sum()}" )
  print(f"\nValores duplicados: {df.duplicated().sum()}")
  print(f"\nPrimeiros registros:\n{df.head()}" )
  print(f"\nÚltimos registros:\n{df.tail()}" )
  print(f"\nEstatísticas descritivas:\n{df.describe()}" )
  return df.describe(include="all")

inspecionar_dados(df)

1. O dataset possui 150 registros e 8 colunas;
2. Há colunas numéricas ('id_venda', 'quantidade', 'preco_unitario') e categóricas ('cliente', 'produto', 'categoria', 'regiao');
3. A coluna 'data_venda' está como object TEXTO e precisará ser convertida para datetime;
4. Foram encontrados valores nulos para as colunas 'quantidade' e 'preco_unitario'. Logo, precisarão ser tratados;
5. Não parece ter outliers significativos;
6. Não há registros duplicados.

In [ ]:
def inspecionar_dados_vendas(df):
    print("\n=== ANÁLISE ESPECÍFICA DO DATASET ===")

    print("\nColunas categóricas:")
    #print(df["cliente"].value_counts())
    print(f"{df["produto"].value_counts()}")
    print(f"\n{df["categoria"].value_counts()}")
    print(f"\n{df["regiao"].value_counts()}")

    print("\nDatas inválidas:")
    print(df[df["data_venda"] == "DATA INVALIDA"])

    print("\nProdutos com espaços extras:")
    print(df[df["produto"] != df["produto"].str.strip()])

inspecionar_dados_vendas(df)

1. Há datas inválidas: DATA INVALIDA;
2. Coluna 'produto' possui valores com espaços em branco.

In [ ]:
df

Será necessário:

1. Tratar valores nulos para as colunas 'quantidade' e 'preco_unitario';
2. Remover linhas com datas inválidas: DATA INVÁLIDA;
3. Remover espaços em branco das strings para a coluna 'produto';
4. Converter 'data_venda' para o tipo datetime.

# RF03 – Limpar e Tratar os Dados

In [ ]:
def remover_espacos_branco(df):
    """
    Remove espaços em branco extras de colunas do tipo texto
    """
    df = df.copy()

    padrao_espacos = re.compile(r"\s+")

    colunas_texto = df.select_dtypes(include="object").columns

    for coluna in colunas_texto:
        df[coluna] = df[coluna].apply(
            lambda texto:
                padrao_espacos.sub(" ", str(texto)).strip()
                if pd.notna(texto) else texto
        )

    return df


def converter_datas(df, colunas):
    """
    Converte colunas para datetime e remove linhas com datas inválidas
    """
    df = df.copy()

    qtd_registros_removidos = 0

    for coluna in colunas:
        df[coluna] = pd.to_datetime(df[coluna], errors="coerce")
        qtd_registros_removidos += df[coluna].isna().sum()

    df = df.dropna(subset=colunas)

    return df, int(qtd_registros_removidos)


def remover_linhas_nulas(df, colunas):
    """
    Remove linhas que possuem nulos nas colunas informadas.
    """
    df = df.copy()
    tamanho_df_antes = len(df)
    df = df.dropna(subset=colunas)
    qtd_registros_removidos = tamanho_df_antes - len(df)

    return df, qtd_registros_removidos


def converter_tipos(df, tipos):
    """
    Converte colunas para os tipos informados.

    Exemplo:
      {"quantidade": int, "preco_unitario": float}
    """
    df = df.copy()
    return df.astype(tipos)


def remover_colunas(df, colunas):
    """
    Remove coluna(s) de um df
    """

    df = df.copy()

    qtd_removidas = len([col for col in colunas if col in df.columns])
    df = df.drop(columns=colunas, errors="ignore")

    return df, qtd_removidas


def gerar_relatorio(titulo, **metricas):
    """
    Gera relatório da limpeza
    """
    relatorio = {}
    relatorio.update(metricas)

    # Exibe relatório
    print(f"\n=== RELATÓRIO - {titulo} ===")
    for chave, valor in relatorio.items():
        print(f"{chave}: {valor}")

    return relatorio


def limpar_dados(df):
    df = remover_espacos_branco(df.copy())

    qtd_inicial = len(df)

    df, qtd_datas_invalidas = converter_datas(df, ["data_venda"])
    df, qtd_nulos_removidos = remover_linhas_nulas(df, ["quantidade", "preco_unitario"])
    df = converter_tipos(df, {"quantidade": int, "preco_unitario": float})
    #df, qtd_colunas_removidas = remover_colunas(df, ["id_venda"])

    relatorio = gerar_relatorio(
        "LIMPEZA DE DADOS",
        qtd_registros_iniciais=qtd_inicial,
        qtd_registros_finais=len(df),
        qtd_registros_removidos_total = qtd_inicial - len(df),
        qtd_datas_invalidas_removidas = qtd_datas_invalidas,
        qtd_linhas_nulas_removidas = qtd_nulos_removidos,
        #qtd_colunas_removidas = qtd_colunas_removidas
    )

    return df, relatorio

def gravar_df_csv(df, diretorio, nome_arquivo):
    """
    Salva o df em CSV.
    """
    os.makedirs(diretorio, exist_ok=True)
    caminho_arquivo = os.path.join(diretorio, nome_arquivo)
    df.to_csv(caminho_arquivo, index=False)
    print(f"\nArquivo salvo em: {caminho_arquivo}")


df_v1, relatorio = limpar_dados(df) # Salva versão limpa/formatada
gravar_df_csv(df_v1, "../data/processed/v1_com_outliers", "vendas_v1.csv")


In [ ]:
df_v1 # Com outliers

In [ ]:
# Conferência do tratamento e limpeza
inspecionar_dados(df_v1)
inspecionar_dados_vendas(df_v1)

A remoção de linhas com valores nulos e datas inválidas não tem impacto significativo no dataset já que são pouquíssimas linhas.

# RF04 – Detectar e Tratar Outliers (versões v1 e v2)

In [ ]:
def tratar_outliers(df, colunas, fator=1.5):
    df = df.copy()

    tamanho_df_antes = len(df)
    limites = {}

    for coluna in colunas:
        q1 = df[coluna].quantile(0.25)
        q3 = df[coluna].quantile(0.75)

        iqr = q3 - q1

        lim_inf = q1 - fator * iqr
        lim_sup = q3 + fator * iqr

        qtd_outliers = ((df[coluna] < lim_inf) | (df[coluna] > lim_sup)).sum()

        limites[coluna] = {
            "outliers": int(qtd_outliers),
            "lim_inf": round(lim_inf, 2),
            "lim_sup": round(lim_sup, 2)
        }

        df = df[(df[coluna] >= lim_inf) & (df[coluna] <= lim_sup)]

    qtd_registros_removidos = tamanho_df_antes - len(df)

    return df, qtd_registros_removidos, limites


def remover_outliers(df):
    df = df.copy()

    tamanho_df_inicial = len(df)

    df["receita_total"] = (df["quantidade"] * df["preco_unitario"])
    df, qtd_outliers_removidos, limites_outliers = tratar_outliers(df, ["quantidade", "receita_total"])
    df, qtd_colunas_removidas = remover_colunas(df, ["receita_total"])

    tamanho_df_atual = len(df)

    relatorio = gerar_relatorio(
        "TRATAMENTO DE OUTLIERS",
        qtd_registros_iniciais=tamanho_df_inicial,
        qtd_registros_finais=tamanho_df_atual,
        qtd_outliers_removidos=qtd_outliers_removidos,
        limites_outliers=limites_outliers,
        qtd_colunas_removidas=qtd_colunas_removidas
    )

    return df, relatorio

df_v2, relatorio = remover_outliers(df_v1) # Salva versão com outliers removidos
gravar_df_csv(df_v2, "../data/processed/v2_outliers_tratado", "vendas_v2.csv")


In [ ]:
grandes_qtds = df_v1[df_v1["quantidade"] > 7]
percentual_grandes_qtds = (
    len(grandes_qtds) / len(df_v1)
) * 100
print(f"Percentual: {percentual_grandes_qtds:.2f}%")

grandes_qtds

O método IQR identificou 6 registros classificados como outliers na variável 'receita_total'. Contudo, após análise, verificou-se que esses registros correspondem a vendas reais de notebooks em grandes quantidades. Considerando que o dataset também apresenta compras em grandes quantidades para outros produtos e que o preço unitário dos notebooks é naturalmente mais alto, essas observações não caracterizam erros ou inconsistências nos dados. Dessa forma, os "outliers" serão mantidos, optando pela versão 1.

# RF05 – Criar Colunas Derivadas com Transformações

In [ ]:
def criar_features(df):
    """
    Cria colunas derivadas.
    """
    df = df.copy()
    qtd_colunas_antes = len(df.columns)

    df["receita_total"] = df["quantidade"] * df["preco_unitario"]
    df["mes"] = df["data_venda"].dt.month
    df["trimestre"] = df["data_venda"].dt.quarter.apply(lambda q: f"Q{q}")
    df["ano"] = df["data_venda"].dt.year

    df["faixa_receita_item"] = np.select(
        [ # Faixas
            df["receita_total"] < 500,
            df["receita_total"] < 5000,
            df["receita_total"] >= 5000
        ],
        [ # Rótulos
            "Baixo Valor",
            "Médio Valor",
            "Alto Valor"
        ],
        default="N/D"
    )

    qtd_colunas_criadas = len(df.columns) - qtd_colunas_antes

    return df, qtd_colunas_criadas


df, qtd_colunas_criadas = criar_features(df_v1) # Passa a tratar como df novamente

relatorio = gerar_relatorio(
    "CRIAÇÃO DE FEATURES",
    qtd_colunas_iniciais=len(df_v1.columns),
    qtd_colunas_finais=len(df.columns),
    qtd_total_colunas_criadas=qtd_colunas_criadas
)

In [ ]:
df

# RF06 – Calcular Métricas Agregadas (groupby)

In [23]:
def metricas_por_mes(df):
  """
  Calcula receita total e quantidade vendida por mês
  """
  return (
      df.groupby("mes").agg(receita_total=("receita_total", "sum"), quantidade=("quantidade", "sum"), nro_vendas=("id_venda", "count"),).reset_index().sort_values("mes"))


def top_produtos(df, top_n=5):
  """
  Calcula receita total por produto (top 5)
  """
  return (df.groupby("produto")["receita_total"].sum().sort_values(ascending=False).head(top_n).reset_index())


def metricas_por_categoria(df):
  """
  Calcula receita total por categoria
  """
  return (df.groupby("categoria")["receita_total"].sum().reset_index().sort_values("receita_total", ascending=False))


def metricas_por_regiao(df):
  """
  Calcula receita total por região
  """
  return (df.groupby("regiao").agg(receita_total=("receita_total", "sum")).reset_index().sort_values("receita_total", ascending=False))


def exibir_metricas(metricas):
  for nome, tabela in metricas.items():
      print(f"\n=== {nome.upper().replace('_', ' ')} ===")
      print(tabela.to_string(index=False))


def calcular_metricas(df):
  metricas = {
      "por_mes": metricas_por_mes(df),
      "top_produtos": top_produtos(df),
      "por_categoria": metricas_por_categoria(df),
      "por_regiao": metricas_por_regiao(df),
  }
  exibir_metricas(metricas)

  return metricas


metricas = calcular_metricas(df)


=== POR MES ===
 mes  receita_total  quantidade  nro_vendas
   1       118690.0          70          12
   2        60840.0          60          12
   3        49050.0          62          11
   4       143040.0          73          13
   5       130890.0          94          16
   6       144940.0          68          10
   7        79780.0          64          13
   8        66280.0          50          12
   9        30280.0          39           8
  10        89450.0          49          11
  11       142950.0          82          14
  12        57760.0          49           8

=== TOP PRODUTOS ===
   produto  receita_total
  Notebook       413000.0
    Tablet       261000.0
Smartphone       257400.0
   Monitor       134400.0
   Teclado        30750.0

=== POR CATEGORIA ===
   categoria  receita_total
Computadores       547400.0
   Celulares       518400.0
 Periféricos        48150.0

=== POR REGIAO ===
      regiao  receita_total
     Sudeste       345390.0
       Norte       289

# RF07 – Segmentar Clientes por Nível de Gasto

In [24]:
def calcular_gasto_clientes(df):
  """
  Calcula total gasto por cada cliente
  """
  return (df.groupby("cliente")["receita_total"].sum().reset_index(name="total_gasto"))


def segmentar_clientes(df):
  """
  Segmenta cliente de acordo com o total gasto
  """
  clientes = calcular_gasto_clientes(df)
  clientes["segmento"] = clientes["total_gasto"].apply(lambda g: "Ouro" if g > 15000 else ("Prata" if g >= 5000 else "Bronze"))

  return clientes.sort_values("total_gasto", ascending=False)


def exibir_segmentacao(clientes, top_n=10):
  print(f"=== SEGMENTAÇÃO DE CLIENTES (Top {top_n}) ===")
  print(clientes.head(top_n).to_string(index=False))

  print("\nDistribuição de segmentos:")
  print(clientes["segmento"].value_counts())


clientes_segmentados = segmentar_clientes(df)
exibir_segmentacao(clientes_segmentados)
clientes_segmentados.head()

=== SEGMENTAÇÃO DE CLIENTES (Top 10) ===
    cliente  total_gasto segmento
Cliente_029      83830.0     Ouro
Cliente_018      67840.0     Ouro
Cliente_014      67160.0     Ouro
Cliente_004      61800.0     Ouro
Cliente_013      61460.0     Ouro
Cliente_003      61200.0     Ouro
Cliente_024      51940.0     Ouro
Cliente_023      51300.0     Ouro
Cliente_009      51150.0     Ouro
Cliente_019      48010.0     Ouro

Distribuição de segmentos:
segmento
Ouro      26
Prata      3
Bronze     1
Name: count, dtype: int64


,cliente,total_gasto,segmento
28,Cliente_029,83830.0,Ouro
17,Cliente_018,67840.0,Ouro
13,Cliente_014,67160.0,Ouro
3,Cliente_004,61800.0,Ouro
12,Cliente_013,61460.0,Ouro
